## Plot Non Cloudflare Public Name Cohort Size - Variant


In [1]:
from pathlib import Path

import altair as alt
import polars as pl

monthYear = "%b %Y"

DATA_DIR = Path("../../data/measurements")

df = pl.read_parquet(DATA_DIR / "ech_cohort_size.parquet")

In [2]:
base = alt.Chart(
    df.filter(pl.col("ech_public_name") != "cloudflare-ech.com")
    .group_by("ech_public_name", "test_date")
    .agg(pl.count("ech_config").alias("count"), pl.col("ech_config_occurrence_count").sum().alias("total_occurrence"))
).encode(
    x=alt.X(
        "test_date:T",
        # title="Date",
        title=None,
        scale=alt.Scale(padding=10),
        axis=alt.Axis(format=monthYear),
    ),
)

# Create the comet chart where test_date is on x-axis, ech_public_name on y-axis
chart_other = (
    base.mark_circle(opacity=0.5)
    .encode(
        y=alt.Y(
            "ech_public_name:N",
            title="ECH Public Name",
            scale=alt.Scale(padding=10),
        ),
        size=alt.Size(
            "total_occurrence:Q",
            title="Total ECHConfig Occurrence",
            # configure legend
            legend=alt.Legend(
                orient="none",
                direction="horizontal",
                legendX=50,
                legendY=285,
            ),
        ),
        color=alt.Color(
            "count:Q",
            title="Distinct ECHConfig Count",
            legend=alt.Legend(orient="none", direction="horizontal", legendX=-135, legendY=285, gradientLength=175),
        ),
    )
    .properties(height=250, width=300)
    .configure_axis(grid=False)
)


chart_other.save("../../plots/ech_distinct_cohort_size.pdf")
chart_other.show()

alt.Chart(...)

## Plot Cloudflare Public Name Cohort Size


In [3]:
COLOR_SCATTER = "#000000"
COLOR_LINE = "#72B7B2"

base = alt.Chart(df.filter(pl.col("ech_public_name") == "cloudflare-ech.com")).encode(
    x=alt.X(
        "test_date:T",
        title=None,
        scale=alt.Scale(padding=10),
        axis=alt.Axis(format="%b %Y"),
    )
)

chart_cloudflare_scatter = base.mark_circle(color=COLOR_SCATTER, opacity=0.6).encode(
    y=alt.Y(
        "ech_config_occurrence_count:Q",
        axis=alt.Axis(
            title="Cohort Size per ECH Config",
            titleColor=COLOR_SCATTER,
        ),
    ),
)

top_hist = base.mark_line(
    color=COLOR_LINE,
    interpolate="monotone",
    point=alt.OverlayMarkDef(color=COLOR_LINE)
).encode(
    y=alt.Y(
        "count()",
        axis=alt.Axis(
            title="Distinct ECH Configs",
            titleColor=COLOR_LINE
        ),
    ),
)

chart_cloudflare = (
    alt.layer(chart_cloudflare_scatter, top_hist)
    .resolve_scale(y="independent")
    .properties(width=300, height=175)
    .configure_axis(
        labelFont="monospace",
        titleFont="monospace",
        labelFontSize=12,
        titleFontSize=12,
        grid=True,
        gridColor="#E8E8E8",
        domainColor="#000000",
        tickColor="#000000"
    )
    .configure_view(
        strokeWidth=0
    )
    .configure_legend(
        labelFont="monospace",
        titleFont="monospace",
        labelFontSize=12,
        titleFontSize=12
    )
)

chart_cloudflare.save("../../plots/ech_cohort_size.pdf")
chart_cloudflare.show()

alt.LayerChart(...)